# 最终 Test 揭盲：已锁定模型的一次性比较

此 Notebook 不训练、不调参、不重选模型。它首先验证全部模型锁定、数据指纹与固定切分，然后仅对最终 Test 一次性评分。输出所有模型的 AUC、AR、KS、AP、Logloss、Top-decile、以及相对共同输入 Logistic 的配对 bootstrap 区间。历史上该随机 Test 已被查看过的限制会写入本次评估清单，因此结果用于固定实验的最终比较，不应被表述为完全新鲜的无偏生产估计。

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'configs' / 'experiment.yaml').is_file()
)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from credit_risk.final_evaluation import run_final_test_evaluation
print('Project root:', PROJECT_ROOT)

Project root: /Users/ganlu/Desktop/home-credit-risk-english


In [2]:
# 当前 WOE bundle 需用兼容环境加载；该文件是原锁定模型导出的、按 ID 对齐的 Test 概率。
woe_prediction_path = PROJECT_ROOT / 'artifacts' / 'woe_compat_test_predictions.csv'
assert woe_prediction_path.is_file()
result = run_final_test_evaluation(PROJECT_ROOT, woe_prediction_path=woe_prediction_path)
print('最终 Test 评分完成。')

最终 Test 评分完成。


In [3]:
display(result.metrics.sort_values('AR', ascending=False))
display(result.uplift_vs_logistic)
display(result.deciles.loc[result.deciles['risk_group'].eq(1)].sort_values('cumulative_bad_capture', ascending=False))
print('最终 Test 产物:', result.run_directory)

,model,n,bad_rate,AUC,AR,KS,AP,Logloss
3,catboost,46127,0.080734,0.770582,0.541164,0.409661,0.256110,0.242643
5,xgboost,46127,0.080734,0.770101,0.540203,0.409830,0.254898,0.242802
4,lightgbm,46127,0.080734,0.767395,0.534791,0.408837,0.250687,0.243692
0,common_logistic,46127,0.080734,0.748776,0.497552,0.374337,0.228022,0.248999
2,random_forest,46127,0.080734,0.748264,0.496529,0.375305,0.229794,0.529193
6,mlp,46127,0.080734,0.748163,0.496326,0.370034,0.225379,0.250509
1,woe_logistic,46127,0.080734,0.742913,0.485827,0.366523,0.219487,0.250663


,model,reference_model,metric,difference,lower,upper,bootstrap_se,confidence_level,bootstrap_replicates,higher_is_better
0,woe_logistic,common_logistic,AUC,-0.005863,-0.008647,-0.003557,0.001306,0.95,500,True
1,woe_logistic,common_logistic,AR,-0.011725,-0.017294,-0.007114,0.002613,0.95,500,True
2,woe_logistic,common_logistic,KS,-0.007814,-0.017213,0.000035,0.004402,0.95,500,True
3,woe_logistic,common_logistic,AP,-0.008535,-0.013675,-0.004140,0.002553,0.95,500,True
4,woe_logistic,common_logistic,Logloss,0.001664,0.000984,0.002416,0.000363,0.95,500,False
5,random_forest,common_logistic,AUC,-0.000512,-0.003561,0.003089,0.001704,0.95,500,True
6,random_forest,common_logistic,AR,-0.001024,-0.007123,0.006177,0.003408,0.95,500,True
7,random_forest,common_logistic,KS,0.000968,-0.008557,0.009630,0.004981,0.95,500,True
8,random_forest,common_logistic,AP,0.001772,-0.003635,0.008051,0.002849,0.95,500,True
9,random_forest,common_logistic,Logloss,0.280194,0.277702,0.282900,0.001378,0.95,500,False


,risk_group,rank_min,rank_max,n,bad_n,bad_rate,probability_min,probability_max,lift,cumulative_bad_capture,model
50,1,1,4613,4613,1285,0.278561,0.183959,0.754608,3.450366,0.345059,xgboost
30,1,1,4613,4613,1281,0.277693,0.179099,0.782907,3.439626,0.343985,catboost
40,1,1,4613,4613,1266,0.274442,0.183111,0.759521,3.399349,0.339957,lightgbm
0,1,1,4613,4613,1218,0.264036,0.176916,0.763439,3.270464,0.327068,common_logistic
20,1,1,4613,4613,1199,0.259918,0.656558,0.919754,3.219447,0.321966,random_forest
60,1,1,4613,4613,1189,0.257750,0.207884,0.736304,3.192596,0.319280,mlp
10,1,1,4613,4613,1181,0.256016,0.176316,0.642841,3.171115,0.317132,woe_logistic


最终 Test 产物: /Users/ganlu/Desktop/home-credit-risk-english/artifacts/final_test/20260831T073511_991324Z
